# Clinic Wait-Time Analytics: Stage 3 Investigation

## Simulated data only
This notebook analyzes only the validated synthetic data in this project. It contains no real clinic, patient, or staff data. Scenario outcomes are **model estimates**, not proven causal effects.

In [ ]:
from pathlib import Path
import runpy
import pandas as pd
from IPython.display import Image, display
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
runpy.run_path(ROOT / 'stage3_analysis.py', run_name='__main__')
METRICS = ROOT / 'data' / 'cleaned' / 'metrics'
IMAGES = ROOT / 'images'
def table(name): return pd.read_csv(METRICS / name)

## Highest-priority operational problems
1. Central City late-morning and late-afternoon congestion.
2. West End opening and mid-morning coverage constraint.
3. Minor Procedure and Chronic Care templates underestimate service time.
4. Model-estimated walkout risk rises materially at 30–44 minutes of waiting.
5. Early and Monday slots concentrate no-shows.

## 1. Overloaded hours, weekdays, and locations
The heatmap identifies repeated high-wait cells, led by Central City and selected West End hours. These are wave-specific problems rather than all-day demand.

**Decision:** shift flexible Central City coverage from the 13:00 low-demand period to 15:00–16:00; protect West End opening coverage before increasing headcount.

In [ ]:
display(Image(filename=str(IMAGES / 'stage3_wait_heatmap_simulated.png')))
table('location_weekday_hour_load_simulated.csv').query('demand_coverage_mismatch == True').head(12)

## 2. Understaffing relative to demand
Required provider equivalents use completed service-hours divided by an 85% utilization target. Central City has the largest recurring average shortfalls at 10:00 and 15:00; West End has smaller shortfalls near 09:00–10:00.

**Decision:** pilot staggered shifts first, then add contingent capacity only if wait targets remain unmet.

In [ ]:
table('staffing_gap_by_location_hour_simulated.csv').sort_values('peak_provider_gap', ascending=False).head(12)

## 3. Service types running longer than scheduled
Booking targets are stated planning assumptions because the source data has no scheduled-duration field. Minor Procedure, Chronic Care, and Primary Care run over target; Vaccination is on target.

**Decision:** add 10 minutes to Minor Procedure bookings and test a five-minute Chronic Care buffer in high-congestion periods.

In [ ]:
display(Image(filename=str(IMAGES / 'stage3_service_duration_variance_simulated.png')))
table('service_duration_variance_simulated.csv')

## 4. Wait range where walkout risk rises
Walkouts do not have a service-start timestamp, so their realized wait is unknown. The chart is a **model estimate** from the simulation's smooth walkout rule. Risk first rises materially at **30–44 minutes** and increases faster beyond 45 minutes.

**Decision:** trigger queue overflow actions at 30 minutes: cross-cover, proactive patient updates, or voluntary rebooking.

In [ ]:
display(Image(filename=str(IMAGES / 'stage3_walkout_wait_model_estimate.png')))
table('walkout_wait_model_estimates_simulated.csv')

## 5. No-show concentration
First-hour slots have the highest simulated no-show rates, with Monday 08:00 also elevated.

**Decision:** use reminder-plus-confirmation outreach for Monday and pre-09:00 bookings; do not broadly overbook before testing completion and walkout effects.

In [ ]:
display(Image(filename=str(IMAGES / 'stage3_no_show_slots_simulated.png')))
table('no_show_by_weekday_slot_simulated.csv').head(12)

## 6. Staffing-demand mismatch
Hourly appointments per provider and wait spikes do not always coincide because queues carry over. Daily staffing totals therefore conceal the operational mismatch.

**Decision:** govern capacity by clinic-date-hour demand, starting with Central City 15:00–16:00 and West End 08:00.

In [ ]:
capacity=table('hourly_capacity_simulated.csv')
capacity.groupby(['clinic_location','hour_of_day'],as_index=False).agg(avg_attended=('attended_appointments','mean'),avg_providers=('providers_scheduled','mean'),avg_appts_per_provider=('appointments_per_provider','mean'),avg_wait=('avg_wait_time_minutes','mean')).sort_values('avg_wait',ascending=False).head(15)

## Recommendation scenarios — model estimates
The scenarios are transparent directional estimates. Reallocation scenarios use observed peak-versus-low-demand wait differences; booking changes use duration variance against stated targets; targeted reminders assume a 15–25% no-show reduction. They are not causal proof.

In [ ]:
table('recommendation_scenarios_model_estimates_simulated.csv')

## Limitations of inference
- Data and embedded patterns are simulated, not real-world evidence.
- Staffing gaps use completed service time and can understate unmet demand from walkouts.
- The model excludes rooms, specialty matching, breaks, emergencies, insurance, and rescheduling.
- Walkout-by-wait findings are model estimates because realized waits are undefined for walkouts.
- Test each intervention with operational guardrails and a pre-defined comparison period.